<a href="https://colab.research.google.com/github/azkakhan4/flyrank-ai-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azkakhan4/flyrank-ai-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method Choice and Why

I will use **Logistic Regression** to build a machine-learning ranking model for prioritising content pages for refresh review.

The target variable will be `is_declining_label`, created from `trend_direction`, where `"down"` is labelled as 1 and all other categories are labelled as 0.

Logistic Regression is suitable because it is an interpretable classification method. It provides prediction scores that can be used to rank content pages.

I will exclude `trend_direction` and `trend_pct` from the input features to prevent direct target leakage.

I will compare the model against the Week 4 rule-based baseline using Precision@50 on the same held-out test data.

**Limitation:** This evaluation measures how well the ranking identifies declining pages. It does not directly measure whether a page has an actual CTR improvement opportunity.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use a grouped train/test split based on `client_id`.

The dataset contains 32 unique clients, with no missing client IDs and no duplicate content-client pairs.

I will keep entire clients in either the training set or the test set. This prevents records from the same client appearing in both sets and helps evaluate how well the model generalizes to unseen clients.

I will use 80% of the clients for training and 20% for testing. The test set will remain separate until final evaluation.

In [12]:
import pandas as pd

url = "https://raw.githubusercontent.com/azkakhan4/flyrank-ai-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))


Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [13]:
print("Unique clients:", df["client_id"].nunique())

print("Missing client IDs:", df["client_id"].isna().sum())

print("Duplicate content-client pairs:",
      df.duplicated(["content_id", "client_id"]).sum())

Unique clients: 32
Missing client IDs: 0
Duplicate content-client pairs: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
from sklearn.model_selection import GroupShuffleSplit

# Create the target label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Split by client_id
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Verify the split
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

print(
    "Overlapping clients:",
    len(set(train_df["client_id"]) & set(test_df["client_id"]))
)

print("\nTraining target distribution:")
print(train_df["is_declining_label"].value_counts())

print("\nTesting target distribution:")
print(test_df["is_declining_label"].value_counts())


Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Overlapping clients: 0

Training target distribution:
is_declining_label
1    13113
0    10724
Name: count, dtype: int64

Testing target distribution:
is_declining_label
1    3149
0    3014
Name: count, dtype: int64


In [15]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

target = "is_declining_label"

print("Selected features:", features)
print("Number of features:", len(features))

print("\nMissing values in training features:")
print(train_df[features].isna().sum())

print("\nMissing values in testing features:")
print(test_df[features].isna().sum())

Selected features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
Number of features: 10

Missing values in training features:
search_volume             2319
competition               2319
cpc                       2319
word_count                6614
char_count                6614
content_age_days             0
days_since_last_update       0
impressions_prev_30d         0
clicks_prev_30d              0
sessions_prev_30d            0
dtype: int64

Missing values in testing features:
search_volume              149
competition                149
cpc                        149
word_count                1085
char_count                1085
content_age_days             0
days_since_last_update       0
impressions_prev_30d         0
clicks_prev_30d              0
sessions_prev_30d            0
dtype: int64


In [16]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Pipeline prepared successfully.")

Training features shape: (23837, 10)
Testing features shape: (6163, 10)
Pipeline prepared successfully.


In [17]:
print("Feature data types:")
print(X_train.dtypes)

print("\nSample feature values:")
print(X_train.head())

Feature data types:
search_volume             float64
competition               float64
cpc                       float64
word_count                float64
char_count                float64
content_age_days            int64
days_since_last_update      int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
dtype: object

Sample feature values:
   search_volume  competition   cpc  word_count  char_count  content_age_days  \
2            0.0         0.00  0.00      3515.0     23643.0               141   
3           10.0         0.00  0.00         NaN         NaN               463   
4            0.0         0.00  0.00      2803.0     17469.0               263   
6            0.0         0.00  0.00      3059.0     20810.0                90   
7          590.0         0.44  0.64         NaN         NaN               445   

   days_since_last_update  impressions_prev_30d  clicks_prev_30d  \
2                      20                  6089

In [18]:
# Train Logistic Regression
model.fit(X_train, y_train)

# Predict probability of decline
model_scores = model.predict_proba(X_test)[:, 1]

print("Model training completed.")
print("Number of test predictions:", len(model_scores))
print("First 10 prediction scores:")
print(model_scores[:10])

Model training completed.
Number of test predictions: 6163
First 10 prediction scores:
[0.58207959 0.39456482 0.65348373 0.60196558 0.60907365 0.61089605
 0.51315187 0.32178673 0.61553729 0.46428675]


In [19]:
# Create a comparison table using the same test records

comparison = test_df[[
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    target
]].copy()

# Logistic Regression ranking
comparison["model_score"] = model_scores

# Week 4 baseline rule
comparison["baseline_score"] = (
    (comparison["days_since_last_update"] >= 90).astype(int) * 2
    + (comparison["impressions_90d"] >= 500).astype(int)
)

# Calculate Precision@50
model_top50 = comparison.nlargest(50, "model_score")
baseline_top50 = comparison.nlargest(50, "baseline_score")

model_precision = model_top50[target].mean()
baseline_precision = baseline_top50[target].mean()

print("MODEL VS BASELINE")
print("-----------------")
print(f"Logistic Regression Precision@50: {model_precision:.4f}")
print(f"Week 4 Baseline Precision@50: {baseline_precision:.4f}")

print("\nModel top-50 declining pages:", model_top50[target].sum())
print("Baseline top-50 declining pages:", baseline_top50[target].sum())

MODEL VS BASELINE
-----------------
Logistic Regression Precision@50: 0.6000
Week 4 Baseline Precision@50: 0.4400

Model top-50 declining pages: 30
Baseline top-50 declining pages: 22


### Model vs Baseline Results

The Logistic Regression model achieved a Precision@50 of **0.60 (60%)**, while the Week 4 rule-based baseline achieved **0.44 (44%)**.

The model identified 30 declining pages among its top 50 recommendations, compared with 22 for the baseline.

The Logistic Regression model performed better by 16 percentage points on this evaluation.

This suggests that the model ranked declining pages more effectively than the baseline on the selected test set. However, the results are limited to the seven held-out clients and do not guarantee the same performance on all clients.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [20]:
# Get the model's top 50 recommendations
top50 = comparison.nlargest(50, "model_score").copy()

# Find incorrect recommendations
false_positives = top50[top50[target] == 0]

print("Incorrect recommendations in model top 50:",
      len(false_positives))

print("\nExamples of incorrect recommendations:")
print(
    false_positives[
        [
            "content_id",
            "model_score",
            "days_since_last_update",
            "impressions_90d",
            target
        ]
    ].head(10).to_string(index=False)
)


Incorrect recommendations in model top 50: 20

Examples of incorrect recommendations:
          content_id  model_score  days_since_last_update  impressions_90d  is_declining_label
content_c84a0ab98e90     0.861383                      20           223271                   0
content_73c54f78c06a     0.840049                      20           213963                   0
content_2db251d1a841     0.786712                      20           198671                   0
content_a5dbb404bdc2     0.779959                     106            79035                   0
content_d0cadc3e2773     0.735007                      89            46437                   0
content_ef9bdd92b523     0.733351                     104                6                   0
content_c9fff7c4660c     0.729628                     104              110                   0
content_dcd38075ec2c     0.727263                     104                3                   0
content_84c5f6af423c     0.723726                     104  

### Error Analysis and Interpretation

The model selected 50 pages with the highest predicted probability of decline. Among these, 30 were correctly identified as declining pages, while 20 were false positives.

Some false positives had very high impressions, while others had low impressions and had not been updated recently. This suggests that the model learned patterns from historical performance and content characteristics that do not always correspond to the declining label.

These errors show that a high predicted probability does not guarantee that a page is actually declining.

The model performed better than the Week 4 baseline on Precision@50, but it still makes incorrect recommendations. Further evaluation using more clients and additional validation would help assess its reliability.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [21]:
# Final project checks

print("1. Target created:", target in df.columns)

print("2. Target leakage columns excluded:",
      "trend_direction" not in features
      and "trend_pct" not in features)

print("3. No overlapping clients:",
      set(train_df["client_id"]).isdisjoint(
          set(test_df["client_id"])
      ))

print("4. Model predictions complete:",
      len(model_scores) == len(test_df))

print("5. Model has 50 recommendations:",
      len(model_top50) == 50)

print("6. Baseline has 50 recommendations:",
      len(baseline_top50) == 50)

print("\nFinal Precision@50:")
print("Model:", round(model_precision, 4))
print("Baseline:", round(baseline_precision, 4))

1. Target created: True
2. Target leakage columns excluded: True
3. No overlapping clients: True
4. Model predictions complete: True
5. Model has 50 recommendations: True
6. Baseline has 50 recommendations: True

Final Precision@50:
Model: 0.6
Baseline: 0.44


### Self-Check Summary

All six verification checks passed successfully.

- The target variable was created correctly.
- Target-related columns were excluded from the input features.
- No clients overlapped between training and testing sets.
- Predictions were generated for all test records.
- Both the model and baseline produced 50 recommendations.
- The Logistic Regression model achieved a Precision@50 of 0.60, compared with 0.44 for the Week 4 baseline.

The model performed better than the baseline on this test set. However, further validation with additional clients is needed before considering it reliable for general use.